## 使用 Bonito 从任何文本中生成SFT数据

需要使用他们的模型 `BatsResearch/bonito-v1`

https://github.com/BatsResearch/bonito/

https://zhuanlan.zhihu.com/p/686856459

然后仓库中各自有一个 在 Colab T4 和 A100 上 demo运行 的 notebook。

这个东西优化一下就可以用了，效果还行。

In [ ]:
# 第 1 步 — 安装 Bonito 包和其他依赖项

# PyMuPDF 用于从 PDF 文件中读取和提取文本
!pip install pymupdf 

# 用于自然语言处理任务   
!pip install spacy      

 #egg=bonito
!pip install -e git+https://github.com/BatsResearch/bonito     

In [ ]:
# 利用 PyMuPDF 库从文档中提取文本

import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)  # Open the PDF file
    text = ""
    for page in doc:  # Iterate through each page
        text += page.get_text()  # Extract text and append it to the text variable
    return text

pdf_path = 'cssf_12_552_governance.pdf'  # Specify the path to your PDF document
text = extract_text_from_pdf(pdf_path)  # Call the function with the path to your PDF

In [ ]:
# 通过将提取的文本分割成句子来处理它。此步骤使用 SpaCy，这是一个用于高级自然语言处理 (NLP) 的库
import spacy

nlp = spacy.load("en_core_web_sm")  # Load the English language model

def split_into_sentences(text):
    doc = nlp(text)  # Process the text with SpaCy
    sentences = [sent.text.strip() for sent in doc.sents]  # Extract sentences and strip whitespace
    return sentences

sentences = split_into_sentences(text)  # Split the extracted text into sentences

In [ ]:
# 将句子列表转换为模型 Bonito 可以使用的格式，特别是使用该datasets库
from datasets import Dataset

# Assuming sentences is a list of strings, where each string is a sentence
data = {"sentence": sentences}
dataset = Dataset.from_dict(data)

print(dataset)

在此示例中，我们使用 Bonito 进行“问题生成”(qg) 来为数据集创建问题。但 Bonito 可以处理多种任务。以下是 Bonito 可以管理的任务类型的简要概述：

提取式问答 (exqa)：根据给定的文本片段生成问题的答案，直接从文本中提取答案。
- 多项选择题问答 (mcqa)：提供一组多项选择中问题的答案。
- 问题生成 (qg)：根据提供的文本内容创建问题。
- 无选择问答 (qa)：回答问题而不提供多项选择选项。
- 是-否问答 (ynqa)：生成问题的是或否答案。
- 共指解析 (coref)：识别文本中引用同一实体的提及。
- 释义生成（paraphrase）：用不同的措辞重写句子或短语，同时保留原始含义。
- 释义识别（paraphrase_id）：确定两个句子或短语是否传达相同的含义。
- 句子完成 (sent_comp)：填写句子中缺失的部分。
- 情绪分析（情绪）：识别文本中表达的情绪，例如积极、消极或中性。
- 摘要：将较长的文本压缩为较短的摘要，抓住要点。
- 文本生成 (text_gen)：根据提示创建连贯且上下文相关的文本。
- 主题分类 (topic_class)：将文本分类为预定义的主题。
- 词义消歧 (wsd)：根据上下文确定单词的含义。
- 文本蕴涵 (te)：预测给定文本是否在逻辑上源自另一文本。
- 自然语言推理 (nli)：确定两段文本之间的关系，例如矛盾、蕴含或中立

In [ ]:
# 生成综合数据集

from bonito import Bonito, SamplingParams
from datasets import load_dataset

# Initialize the Bonito model
bonito = Bonito("BatsResearch/bonito-v1")

sampling_params = SamplingParams(max_tokens=256, top_p=0.95, temperature=0.5, n=1)
synthetic_dataset = bonito.generate_tasks(
    dataset,
    context_col="sentence",
    task_type="qg", # 问题生成 
    sampling_params=sampling_params
)

In [ ]:
# 保存生成的数据集

from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# 然后为数据集创建存储库并将其推送到中心。

from huggingface_hub import create_repo
from huggingface_hub import Repository

repo_name = "dataset_12_552"  # Choose a name for your dataset repository
repo_url = create_repo(repo_name, repo_type="dataset")
print("Repository URL:", repo_url)
synthetic_dataset.push_to_hub(f"Ronal999/dataset_12_552")